In [ ]:
from signal import default_int_handler, signal

import matplotlib.pyplot as plt
from IPython.display import clear_output
from nn import *


class Seqs(Sequential):
    def forward(s, x):
        return tuple(s(x := S(x)) for S, s in zip(s[::2], s[1::2]))


def Log(*A, f=torch.log, g=np.exp, N=100):
    X = n = 0
    a = []
    A = np.empty((0,) + A, "f")

    def Log(x=None):
        nonlocal X, n, A
        if x is None:
            if a:
                A = np.concatenate((A, g(np.stack(a) / N)))
                a.clear()
            return A
        X += f(x)
        n += 1
        if n >= N:
            a.append(X.to("cpu", torch.float, True).numpy())
            X = n = 0

    return Log


dat = Dat("../dat/py.txt", 1, 128, 65)

In [ ]:
decay = torch.logspace(-4, 4, 63, dtype=torch.bfloat16, device="cuda")[:, None, None, None]
Decay = 1 / decay.cumsum(-4)
i, h, k, v, a = 64, 64, 4, 4, 256
m = Seq(
    Embedding(68, i),
    Seqs(
        *(
            _
            for _ in range(8)
            for _ in (
                Add(
                    Seq(
                        Add(
                            Seq(
                                Dot(
                                    Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                    Seq(
                                        Mul(
                                            Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                            Seq(Linear(i, h * v), Unflatten(-1, (h, v, 1))),
                                        ),
                                        F(lambda x: functional.pad((x[..., :-1, :, :, :] * decay).cumsum(-4) * Decay, (0, 0, 0, 0, 0, 0, 1, 0))),
                                    ),
                                ),
                                Flatten(-3),
                                Linear(h * v, a, False),
                            ),
                            Linear(i, a, False),
                        ),
                        T(BatchNorm1d(a, track_running_stats=False)),
                        SiLU(True),
                        Linear(a, i, False),
                    ),
                    Linear(i, i, False),
                ),
                Linear(i, 68),
            )
        )
    ),
).to("cuda", torch.bfloat16, True)
m = torch.compile(torch.jit.trace(m, dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 2.5e-4, fused=True)
L = 9e9
logl = Log(8)
loga = Log(8)
loglr = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    y = m(x[:, :-1])
    x = x[:, 1:]
    ls = torch.stack([functional.cross_entropy(y.mT, x) for y in y])
    l = ls.mean()
    o.zero_grad()
    l.backward()
    o.step()
    l = l.data.float()
    o.param_groups[0]["lr"] *= 0.999 ** (L - l).sign()
    L = l
    logl(ls.data)
    loga(torch.stack([(y.argmax(2) != x).sum() / x.numel() for y in y]))
    loglr(o.param_groups[0]["lr"])
    if not (i + 1) % 1000 or sigint:
        plt.figure(figsize=(12, 12))
        plt.subplot(311)
        for i in logl().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(312)
        for i in loga().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(313).plot(loglr(), label=f"{loglr().min():.1e} {loglr()[-1]:.1e}")
        plt.yscale("log")
        plt.legend()
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
decay = torch.logspace(-4, 4, 63, dtype=torch.bfloat16, device="cuda")[:, None, None, None]
Decay = 1 / decay.cumsum(-4)
i, h, k, v, a = 64, 64, 4, 4, 256
m = Seq(
    Embedding(68, i),
    Seqs(
        *(
            _
            for _ in range(8)
            for _ in (
                Add(
                    Seq(
                        Add(
                            Seq(
                                Dot(
                                    Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                    Seq(
                                        Mul(
                                            Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                            Seq(Linear(i, h * v), Unflatten(-1, (h, v, 1))),
                                        ),
                                        F(lambda x: functional.pad((x[..., :-1, :, :, :] * decay).cumsum(-4) * Decay, (0, 0, 0, 0, 0, 0, 1, 0))),
                                    ),
                                ),
                                Flatten(-3),
                                Linear(h * v, i, False),
                            ),
                            Linear(i, i, False),
                        ),
                        T(BatchNorm1d(i, track_running_stats=False)),
                        SiLU(True),
                    ),
                    Linear(i, i, False),
                ),
                Linear(i, 68),
            )
        )
    ),
).to("cuda", torch.bfloat16, True)
m = torch.compile(torch.jit.trace(m, dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 2.5e-4, fused=True)
L = 9e9
logl = Log(8)
loga = Log(8)
loglr = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    y = m(x[:, :-1])
    x = x[:, 1:]
    ls = torch.stack([functional.cross_entropy(y.mT, x) for y in y])
    l = ls.mean()
    o.zero_grad()
    l.backward()
    o.step()
    l = l.data.float()
    o.param_groups[0]["lr"] *= 0.999 ** (L - l).sign()
    L = l
    logl(ls.data)
    loga(torch.stack([(y.argmax(2) != x).sum() / x.numel() for y in y]))
    loglr(o.param_groups[0]["lr"])
    if not (i + 1) % 1000 or sigint:
        plt.figure(figsize=(12, 12))
        plt.subplot(311)
        for i in logl().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(312)
        for i in loga().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(313).plot(loglr(), label=f"{loglr().min():.1e} {loglr()[-1]:.1e}")
        plt.yscale("log")
        plt.legend()
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
i, h, k, v, a = 256, 16, 16, 16, 256
m = Seq(
    Embedding(68, i),
    Seqs(
        *(
            _
            for _ in range(8)
            for _ in (
                Add(
                    Seq(
                        Add(
                            Seq(
                                Dot(
                                    Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                    Seq(
                                        Mul(
                                            Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                            Seq(Linear(i, h * v), Unflatten(-1, (h, v, 1))),
                                        ),
                                        Decay((h, v, 1), (h, 1, k), d=-4, shift=1, min=2, max=8),
                                    ),
                                ),
                                Flatten(-3),
                                Linear(h * v, i, False),
                            ),
                            Linear(i, i, False),
                        ),
                        T(BatchNorm1d(i, track_running_stats=False)),
                        SiLU(True),
                    ),
                    Linear(i, i, False),
                ),
                Linear(i, 68),
            )
        )
    ),
).to("cuda", torch.bfloat16, True)
m = torch.compile(torch.jit.trace(m, dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 2.5e-4, fused=True)
L = 9e9
logl = Log(8)
loga = Log(8)
loglr = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    y = m(x[:, :-1])
    x = x[:, 1:]
    ls = torch.stack([functional.cross_entropy(y.mT, x) for y in y])
    l = ls.mean()
    o.zero_grad()
    l.backward()
    o.step()
    l = l.data.float()
    o.param_groups[0]["lr"] *= 0.999 ** (L - l).sign()
    L = l
    logl(ls.data)
    loga(torch.stack([(y.argmax(2) != x).sum() / x.numel() for y in y]))
    loglr(o.param_groups[0]["lr"])
    if not (i + 1) % 1000 or sigint:
        plt.figure(figsize=(12, 12))
        plt.subplot(311)
        for i in logl().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(312)
        for i in loga().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(313).plot(loglr(), label=f"{loglr().min():.1e} {loglr()[-1]:.1e}")
        plt.yscale("log")
        plt.legend()
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
i, h, k, v, a = 64, 16, 16, 16, 256
decay = torch.logspace(-4, 4, 63, dtype=torch.bfloat16, device="cuda")[:, None, None, None]
decay_ = 1 / decay.cumsum(-4)
m = Seq(
    Embedding(68, i, scale_grad_by_freq=True),
    Seqs(
        *(
            _
            for _ in range(8)
            for _ in (
                Add(
                    Seq(
                        Add(
                            Seq(
                                Dot(
                                    Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                    Seq(
                                        Mul(
                                            Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                            Seq(Linear(i, h * v), Unflatten(-1, (h, v, 1))),
                                        ),
                                        F(lambda x: functional.pad((x[..., :-1, :, :, :] * decay).cumsum(-4) * decay_, (0, 0, 0, 0, 0, 0, 1, 0))),
                                    ),
                                ),
                                Flatten(-3),
                                Linear(h * v, a, False),
                            ),
                            Linear(i, a, False),
                        ),
                        T(BatchNorm1d(a, track_running_stats=False)),
                        SiLU(True),
                        Linear(a, i, False),
                    ),
                    Linear(i, i, False),
                ),
                Linear(i, 68),
            )
        )
    ),
).to("cuda", torch.bfloat16, True)
m = torch.compile(torch.jit.trace(m, dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-4, fused=True)
L = 9e9
logl = Log(8)
loga = Log(8)
loglr = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    y = m(x[:, :-1])
    x = x[:, 1:]
    ls = torch.stack([functional.cross_entropy(y.mT, x) for y in y])
    l = ls.mean()
    o.zero_grad()
    l.backward()
    o.step()
    l = l.data.float()
    o.param_groups[0]["lr"] *= 0.999 ** (L - l).sign()
    L = l
    logl(ls.data)
    loga(torch.stack([(y.argmax(2) != x).sum() / x.numel() for y in y]))
    loglr(o.param_groups[0]["lr"])
    if not (i + 1) % 1000 or sigint:
        plt.figure(figsize=(12, 12))
        plt.subplot(311)
        for i in logl().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(312)
        for i in loga().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(313).plot(loglr(), label=f"{loglr().min():.1e} {loglr()[-1]:.1e}")
        plt.yscale("log")
        plt.legend()
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)

In [ ]:
i, h, k, v, a = 64, 16, 16, 16, 256
decay = torch.logspace(-8, 8, 63, dtype=torch.bfloat16, device="cuda")[:, None, None, None]
decay_ = 1 / decay.cumsum(-4)
m = Seq(
    Embedding(68, i, scale_grad_by_freq=True),
    Seqs(
        *(
            _
            for _ in range(8)
            for _ in (
                Add(
                    Seq(
                        Add(
                            Seq(
                                Dot(
                                    Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                    Seq(
                                        Mul(
                                            Seq(Linear(i, h * k), Unflatten(-1, (h, 1, k))),
                                            Seq(Linear(i, h * v), Unflatten(-1, (h, v, 1))),
                                        ),
                                        F(lambda x: functional.pad((x[..., :-1, :, :, :] * decay).cumsum(-4) * decay_, (0, 0, 0, 0, 0, 0, 1, 0))),
                                    ),
                                ),
                                Flatten(-3),
                                Linear(h * v, a, False),
                            ),
                            Linear(i, a, False),
                        ),
                        T(BatchNorm1d(a, track_running_stats=False)),
                        SiLU(True),
                        Linear(a, i, False),
                    ),
                    Linear(i, i, False),
                ),
                Linear(i, 68),
            )
        )
    ),
).to("cuda", torch.bfloat16, True)
m = torch.compile(torch.jit.trace(m, dat()[:, :-1]))
o = torch.optim.Adam(m.parameters(), 1e-4, fused=True)
L = 9e9
logl = Log(8)
loga = Log(8)
loglr = Log()
sigint = False
signal(2, lambda *_: globals().__setitem__("sigint", True))
for i, x in enumerate(dat):
    y = m(x[:, :-1])
    x = x[:, 1:]
    ls = torch.stack([functional.cross_entropy(y.mT, x) for y in y])
    l = ls.mean()
    o.zero_grad()
    l.backward()
    o.step()
    l = l.data.float()
    o.param_groups[0]["lr"] *= 0.999 ** (L - l).sign()
    L = l
    logl(ls.data)
    loga(torch.stack([(y.argmax(2) != x).sum() / x.numel() for y in y]))
    loglr(o.param_groups[0]["lr"])
    if not (i + 1) % 1000 or sigint:
        plt.figure(figsize=(12, 12))
        plt.subplot(311)
        for i in logl().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(312)
        for i in loga().T:
            plt.plot(i, label=f"{i.min()*1000:.0f} {i[-1]*1000:.0f}")
        plt.yscale("log")
        plt.legend()
        plt.subplot(313).plot(loglr(), label=f"{loglr().min():.1e} {loglr()[-1]:.1e}")
        plt.yscale("log")
        plt.legend()
        clear_output(True)
        plt.show()
        if sigint:
            break
signal(2, default_int_handler)